# Train a demo version of the control AI on Google Colab (about 30–60 minutes)

1. Menu **Runtime → Change runtime type → GPU (a T4 is enough)**
2. Run the cells below in order. The first cell asks you to upload `groh2024_repro_kit.zip`.
3. At the end, `checkpoint.zip` is downloaded. Unzip it into the local toolkit's `dls/checkpoint/` folder and use `dls/predict.py` for inference during the demo.

Note: the paper's control AI was also trained on the authors' private textbook/atlas images (31,219 images in total). Only the public Fitzpatrick17k is used here, so the accuracy will not match the paper; this is a demo model for explaining the training process.

In [ ]:
from google.colab import files
up = files.upload()            # choose groh2024_repro_kit.zip
!unzip -q -o groh2024_repro_kit.zip
%cd groh2024_repro_kit
!nvidia-smi -L

In [ ]:
# Fitzpatrick17k label table (if the link is dead, download it manually from https://github.com/mattgroh/fitzpatrick17k and upload it)
!wget -q -O fitzpatrick17k.csv https://raw.githubusercontent.com/mattgroh/fitzpatrick17k/main/fitzpatrick17k.csv
!head -3 fitzpatrick17k.csv
# 9 classes (8 main conditions + Other), at most 300 images each; dead original links are skipped automatically
!python dls/fetch_f17k_images.py --csv fitzpatrick17k.csv --mode training --per-class 300 --out-dir f17k_images --train-csv f17k_train.csv

In [ ]:
!python dls/train_control_dls.py --csv f17k_train.csv --img-root f17k_images --epochs 6 --batch-size 64

In [ ]:
# Wizard of Oz: randomly correct 65% of the wrong validation-set predictions to get the enhanced AI
!python dls/wizard_of_oz.py --pred dls/checkpoint/val_predictions.csv --out dls/checkpoint/treatment_predictions.csv

In [ ]:
!zip -q -r checkpoint.zip dls/checkpoint
files.download('checkpoint.zip')